# Session 21 — Real-Time IoT Predictive Maintenance using MLOps

**Goal:** predict equipment failure *before* it happens from streaming sensor data —
train a model on simulated machine telemetry (temperature, vibration, pressure), and
build a real-time scoring loop that flags a machine for maintenance as new readings
arrive.

## The predictive maintenance framing

Unlike a one-shot classification (Sessions 1, 15), IoT predictive maintenance is
inherently a **streaming** problem: readings arrive continuously, and the further out
you can predict a failure, the more useful the warning. This notebook trains a model
to predict "will this machine fail within the next N readings?" using a rolling
window of recent sensor values as features.

## Prerequisites

```bash
pip install mlflow scikit-learn
```
Runs entirely locally with a synthetic sensor dataset (no real IoT hardware/fleet
needed — the resulting model/pipeline shape transfers directly to real sensor data).

In [ ]:
import numpy as np
import pandas as pd
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, precision_recall_curve

mlflow.set_tracking_uri("mlruns")
mlflow.set_experiment("session21-iot-predictive-maintenance")

## Step 1 — Simulate a fleet of machines with degrading sensors

Each machine runs for a random lifetime; sensor readings drift upward (temperature,
vibration) as it approaches failure, with noise layered on top — a standard synthetic
pattern for prototyping predictive maintenance before real sensor data is available.

In [ ]:
rng = np.random.default_rng(42)
N_MACHINES = 60
records = []

for machine_id in range(N_MACHINES):
    lifetime = rng.integers(100, 400)
    base_temp = rng.normal(70, 3)
    base_vibration = rng.normal(0.5, 0.1)
    base_pressure = rng.normal(30, 2)

    for t in range(lifetime):
        degradation = (t / lifetime) ** 2  # accelerating wear near end of life
        temp = base_temp + degradation * 25 + rng.normal(0, 1.5)
        vibration = base_vibration + degradation * 1.2 + rng.normal(0, 0.05)
        pressure = base_pressure - degradation * 8 + rng.normal(0, 1)

        will_fail_soon = int(t >= lifetime - 10)  # fails within next 10 readings

        records.append({
            "machine_id": machine_id, "t": t, "temperature": temp,
            "vibration": vibration, "pressure": pressure, "will_fail_soon": will_fail_soon,
        })

sensor_df = pd.DataFrame(records)
print(f"{len(sensor_df)} readings across {N_MACHINES} machines")
print(f"Positive rate (will fail within 10 readings): {sensor_df['will_fail_soon'].mean():.2%}")

## Step 2 — Feature engineering: rolling statistics, not raw readings

A single instantaneous reading is noisy; a **rolling window** (mean, std, trend over
the last K readings) captures the *trajectory* toward failure, which is what
actually predicts an upcoming failure better than any single point.

In [ ]:
WINDOW = 10

def add_rolling_features(df):
    df = df.sort_values(["machine_id", "t"]).copy()
    for col in ["temperature", "vibration", "pressure"]:
        grouped = df.groupby("machine_id")[col]
        df[f"{col}_roll_mean"] = grouped.transform(lambda s: s.rolling(WINDOW, min_periods=1).mean())
        df[f"{col}_roll_std"] = grouped.transform(lambda s: s.rolling(WINDOW, min_periods=1).std().fillna(0))
        df[f"{col}_trend"] = grouped.transform(lambda s: s.diff(WINDOW).fillna(0))
    return df

featured_df = add_rolling_features(sensor_df)
feature_cols = [c for c in featured_df.columns if "_roll_" in c or "_trend" in c] + \
                ["temperature", "vibration", "pressure"]
print(f"{len(feature_cols)} features:", feature_cols)

## Step 3 — Train, splitting by machine (not by row)

Splitting rows randomly would leak a machine's near-failure readings into both
train and test (since consecutive readings are highly correlated) — split by
`machine_id` instead, so the model is evaluated on machines it has never seen.

In [ ]:
machine_ids = featured_df["machine_id"].unique()
train_ids, test_ids = train_test_split(machine_ids, test_size=0.25, random_state=0)

train_df = featured_df[featured_df["machine_id"].isin(train_ids)]
test_df = featured_df[featured_df["machine_id"].isin(test_ids)]

X_train, y_train = train_df[feature_cols], train_df["will_fail_soon"]
X_test, y_test = test_df[feature_cols], test_df["will_fail_soon"]

with mlflow.start_run(run_name="predictive_maintenance_gbm") as run:
    model = GradientBoostingClassifier(n_estimators=150, max_depth=3, random_state=0)
    model.fit(X_train, y_train)

    proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, proba)

    mlflow.log_param("window_size", WINDOW)
    mlflow.log_param("n_estimators", 150)
    mlflow.log_metric("test_auc", auc)
    mlflow.sklearn.log_model(model, artifact_path="model")

    run_id = run.info.run_id
    print(f"test AUC: {auc:.4f}  (split by machine_id, not by row)")

## Step 4 — Pick an alert threshold using precision/recall trade-off

For maintenance alerts, missing a real failure (false negative) usually costs far
more than an unnecessary inspection (false positive) — pick a threshold that favors
recall, not the default 0.5.

In [ ]:
precisions, recalls, thresholds = precision_recall_curve(y_test, proba)

target_recall = 0.90
idx = np.argmin(np.abs(recalls[:-1] - target_recall))
chosen_threshold = thresholds[idx]
print(f"At recall={recalls[idx]:.2f}, precision={precisions[idx]:.2f}, threshold={chosen_threshold:.3f}")
print(f"(default 0.5 threshold would give precision={precisions[np.argmin(np.abs(thresholds-0.5))]:.2f}, "
      f"recall={recalls[np.argmin(np.abs(thresholds-0.5))]:.2f})")

## Step 5 — Real-time scoring loop

Simulate streaming: readings arrive one at a time for a machine, rolling features are
recomputed incrementally, and each new reading gets scored immediately — this is the
shape a real Kafka-consumer or MQTT-subscriber scoring service would follow.

In [ ]:
import mlflow.sklearn

loaded_model = mlflow.sklearn.load_model(f"runs:/{run_id}/model")

def score_stream(machine_readings_df, threshold=chosen_threshold):
    featured = add_rolling_features(machine_readings_df)
    alerts = []
    for _, row in featured.iterrows():
        proba = loaded_model.predict_proba(row[feature_cols].values.reshape(1, -1))[0, 1]
        if proba >= threshold:
            alerts.append((int(row["t"]), round(float(proba), 3)))
    return alerts

sample_machine = sensor_df[sensor_df["machine_id"] == test_ids[0]]
alerts = score_stream(sample_machine)
print(f"Machine {test_ids[0]} (lifetime={len(sample_machine)} readings):")
print(f"First alert at reading t={alerts[0][0]} (probability={alerts[0][1]}), "
      f"true failure window starts at t={len(sample_machine)-10}")
print(f"Total alerts: {len(alerts)} of {len(sample_machine)} readings")

## What to try next

* Replace `GradientBoostingClassifier` with a sequence model (an LSTM, following the
  same `nn.LSTM` pattern as the text-generation notebook in
  `10. NLP/Deep learning/7. LLM Pytorch/3.DeepLearningforText/text_generation.ipynb`)
  to let the model learn temporal patterns directly instead of hand-engineered
  rolling features.
* Feed `score_stream`'s alerts into Session 12's BigQuery-style monitoring table, and
  track *lead time* (how many readings before actual failure the first alert fired)
  as the key business metric, not just AUC.
* Wrap the scoring function in the FastAPI pattern from Session 7 so a real message
  queue consumer can call it per-message.